In [0]:
import yaml
from pyspark.sql import functions as F

config_path = "../conf/base_config.yaml"

try:
    with open(config_path, "r") as file:
        config = yaml.safe_load(file)
except FileNotFoundError:
    raise Exception(f"Arquivo de configuração não encontrado no caminho: {config_path}")

paysim_bronze = config['paths']['bronze']
paysim_silver = config['paths']['silver']

print(f"Origem (Bronze): {paysim_bronze}")
print(f"Destino (Silver): {paysim_silver}")

In [0]:
print("Lendo dados da camada Bronze...")
df_bronze = spark.read.table(paysim_bronze)

In [0]:
print("Casting e camada Silver...")

spark.sql(f"""
CREATE OR REPLACE TABLE {paysim_silver}
USING DELTA
PARTITIONED BY (step)
AS
SELECT DISTINCT
    CAST(step AS INT) as step,
    CAST(type AS STRING) as type,
    CAST(amount AS DOUBLE) as amount,
    CAST(nameOrig AS STRING) as name_orig,
    CAST(oldbalanceOrg AS DOUBLE) as old_balance_org,
    CAST(newbalanceOrig AS DOUBLE) as new_balance_orig,
    CAST(nameDest AS STRING) as name_dest,
    CAST(oldbalanceDest AS DOUBLE) as old_balance_dest,
    CAST(newbalanceDest AS DOUBLE) as new_balance_dest,
    CAST(isFraud AS INT) as is_fraud,
    CAST(isFlaggedFraud AS INT) as is_flagged_fraud
    
FROM {paysim_bronze}
""")

print("Camada Bronze -> Silver processada com sucesso!")

In [0]:
print("Lendo dados da camada Silver...")
df_silver = spark.read.table(paysim_silver)

In [0]:
# Remoção de Duplicatas com log

before = df_silver.count()
df_silver = df_silver.dropDuplicates()
after = df_silver.count()

dduplicates = before - after

if dduplicates > 0:
    print(f"⚠️ Foram encontradas e removidas {dduplicates} linhas duplicadas.")
else:
    print("✅ Nenhuma linha duplicada encontrada.")

In [0]:
print(f"Salvando dados na tabela Silver ({paysim_silver})...")

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("step") \
    .saveAsTable(paysim_silver)

print("Camada Silver -> Pré Processamento Concluído!")